<a href="https://colab.research.google.com/github/jwcunhagit/phd_ufpe/blob/main/toolkit_bias_fairness_p1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Toolkit de Bias e Fairness em IA: Teoria e Prática (Parte 1/3)

> Material produzido para a disciplina 'Ética em Inteligência Artificial' ministrada pelo Prof. Geber Ramalho no Mestrado em Ciência da Computação do Centro de Informática da Universidade Federal de Pernambuco (CIn - UFPE)

Projeto finalizado e entregue em 16/12/25.

Discentes:

* Letícia Leitão ([github](https://github.com/leticialevleitao)/[linkedin](https://www.linkedin.com/in/leticialevleitao/))
* Renato Samico ([github](https://github.com/dsnato)/[linkedin](https://))

##Sobre o Toolkit
O Toolkit foi projetado com uma estrutura progressiva de aprendizagem, organizada em três níveis de complexidade. Este notebook serve como introdução ao estudo e aplicação de conceitos de fairness (equidade algorítmica) em machine learning (aprendizado de máquina). Desenvolvido para execução no Google Colab, combina fundamentos teóricos com exemplos práticos implementáveis.

Os próximos notebooks (Parte 2 e 3) complementarão a formação teórica e prática necessária para um entendimento abrangente das possibilidades de análise e mitigação de viés neste campo.

###Como usar?
**Leia as células de texto** que acompanham os trechos de código para se familiarizar com o contexto e em seguida **aperte ▶ para rodar a célula ao vivo**

##Objetivos de Aprendizado (3 Notebooks)
###Conceitos fundamentais
- Compreender o que é viés em IA e como ele surge

- Identificar diferentes tipos de vieses (dados, modelo, medição)

- Analisar casos reais em setores críticos

- Aplicar técnicas de mitigação práticas

###Aplicações reais
- Justiça criminal ([Sistema COMPAS](https://en.wikipedia.org/wiki/COMPAS_(software)))

- Sistemas de crédito ([Risco de crédito Alemão](https://www.kaggle.com/datasets/uciml/german-credit))

- Disparidades em saúde

- Processos de recrutamento

###Ferramentas práticas
- AIF360 - Toolkit completo da IBM ([saiba mais](https://aif360.readthedocs.io/en/stable/Getting%20Started.html))

- FairLearn - Biblioteca Microsoft para fairness ([saiba mais](https://fairlearn.org/))

- WHAT-IF Tool - Análise visual interativa do Google ([saiba mais](https://research.google/pubsthe-what-if-tool-interactive-probing-of-machine-learning-models/))

###Atividade Prática
- Mão na massa: análise e execução de código.

- Debate das respostas na próxima aula.

## 1. O Que é viés em IA e como ele surge
###Definição
Viés em IA refere-se a resultados sistemáticos e injustos que favorecem ou prejudicam certos grupos de pessoas. São padrões discriminatórios aprendidos nas leituras de conjuntos de dados e reproduzidos por sistemas de machine learning.

##Como os Vieses Surgem

In [ ]:
# Exemplo simples de viés algorítmico
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
n_samples = 1000

dados = {
    'grupo': np.random.choice(['A', 'B'], n_samples, p=[0.7, 0.3]),
    'pontuacao': np.random.normal(0, 1, n_samples)
}

# Introduzindo viés sistemático
df = pd.DataFrame(dados)
df['aprovado'] = ((df['pontuacao'] +
                  (df['grupo'] == 'B') * -1.0 +  # Grupo B penalizado por razão externa
                  np.random.normal(0, 0.5, n_samples)) > 0).astype(int)

print("Taxa de aprovação com viés:\n")
print(df.groupby('grupo').mean())

Taxa de aprovação com viés:

       pontuacao  aprovado
grupo                     
A       0.151189  0.553371
B      -0.030386  0.215278


### Explicando o que aconteceu:
- A linha de código responsável pela introdução do viés:

> `df['aprovado'] = ((df['pontuacao'] + (df['grupo'] == 'B') * -1.0 + np.random.normal(0, 0.5, n_samples)) > 0).astype(int)`



- Nesta linha, a parte crucial é `(df['grupo'] == 'B') * -1.0`.
- Funciona da seguinte forma: *` df['grupo'] == 'B'` cria uma série booleana (Verdadeiro/Falso) que é `True` para os indivíduos do Grupo B e `False` para os do Grupo A.
- Quando você multiplica um booleano por um número, `True` é tratado como 1 e `False` como 0.
- Assim, para todos os indivíduos do Grupo B,` (True * -1.0)` resulta em -1.0.
- Para os indivíduos do Grupo A, `(False * -1.0)` resulta em 0.
- Isso significa que, antes mesmo de considerar a aleatoriedade `(np.random.normal)`, os indivíduos do Grupo B têm 1.0 subtraído de sua pontuação base `(df['pontuacao'])`, enquanto os do Grupo A não têm nada subtraído.

- **Conclusão**: essa **penalidade sistemática** para o Grupo B torna muito mais difícil para eles atinjam uma pontuação total maior que zero, o que é o critério para serem 'aprovados'.

> - No mundo real, isso pode ser comparado a situações em que determinadas características presentes no histórico de uma pessoa são interpretadas como negativas. Como consequência, o score final desse indivíduo é reduzido. Exemplos de tais características incluem histórico criminal, nível de renda familiar, orientação sexual, formação acadêmica, entre outros.

Imagine que para ser '**aprovado**', você precisa ter uma pontuação final acima de zero (>0). Todo mundo começa com uma pontuação base (que é o `df['pontuacao']` e uma pequena **variação aleatória**). O que fizemos no código foi o seguinte:

1. Pontuação base: Todo mundo tem uma pontuação inicial, como se fosse um ponto de partida.
2. **Penalidade para o Grupo B:** Automaticamente, para qualquer pessoa que pertencia ao 'Grupo B', nós subtraímos um ponto inteiro da sua pontuação final. O **'Grupo A' não sofreu essa subtração**.
3. Resultado: Com essa penalidade, mesmo que um membro do Grupo B tivesse uma pontuação inicial boa, ele teve que superar essa desvantagem de um ponto. O Grupo A não teve essa desvantagem.

**Por que isso é um viés?**

É um viés porque a decisão de 'aprovar' ou 'não aprovar' não foi baseada apenas no mérito individual (a pontuação base), mas sim na pertença a um grupo. O Grupo B foi penalizado apenas por ser o Grupo B, tornando muito mais difícil para seus membros serem aprovados em comparação com o Grupo A, mesmo que tivessem características iniciais semelhantes. Isso leva a resultados desiguais e injustos entre os grupos.

### Principais fontes de viés:

* **Dados históricos**: Desigualdades passadas são capturadas nos dados

* **Amostragem não representativa**: Grupos minoritários sub-representados

* **Proxies de atributos sensíveis**: Variáveis correlacionadas com raça, gênero, idade

* **Design do algoritmo**: Otimização por métricas que ignoram fairness

* **Viés do desenvolvedor**: Suposições inconscientes no design do sistema

## 2. Principais Tipos de Vieses
###2.1 Viés de amostragem não representativa (dados)

In [ ]:
def demonstrar_vies_dados():
    # Viés de representação
    np.random.seed(42)

    # População real: 50% Grupo A, 50% Grupo B
    # Amostra coletada: 80% Grupo A, 20% Grupo B
    tamanho_amostra = 1000
    grupo_real = np.random.choice(['A', 'B'], tamanho_amostra, p=[0.5, 0.5])

    # Coleta enviesada (Grupo A mais disponível)
    probabilidade_coleta = np.where(grupo_real == 'A', 0.8, 0.2)
    foi_coletado = np.random.binomial(1, probabilidade_coleta)

    dados_coletados = pd.DataFrame({
        'grupo': grupo_real[foi_coletado == 1],
        'feature': np.random.normal(0, 1, sum(foi_coletado))
    })

    print("Viés de Representação nos Dados Coletados:")
    print(f"População real: 50% A, 50% B\n")
    print(f"Amostra coletada-> {dados_coletados['grupo'].value_counts(normalize=True)}")

    return dados_coletados

dados_viesados = demonstrar_vies_dados()

Viés de Representação nos Dados Coletados:
População real: 50% A, 50% B

Amostra coletada-> grupo
A    0.804481
B    0.195519
Name: proportion, dtype: float64


### O que foi feito?

* **Primeiro**, foi criada uma população fictícia composta igualmente por dois grupos (50% A e 50% B) -> `grupo_real = np.random.choice(['A', 'B'], tamanho_amostra, p=[0.5, 0.5])`.

* Em seguida, simulou-se um **processo de coleta enviesado**, no qual indivíduos do **Grupo A tinham 80% de chance de serem incluídos** na amostra, enquanto os do **Grupo B tinham apenas 20%**. Esse desequilíbrio faz com que a **amostra coletada não reflita fielmente a composição da população real**, resultando em uma **predominância do Grupo A nos dados finais.**  

**Por que isso é um viés?**
> Esse fenômeno caracteriza um **viés de representatividade**, pois um dos grupos aparece de forma **sub-representada nos dados**. Na prática, isso significa que qualquer modelo ou análise construída sobre essa amostra terá **maior influência das características do Grupo A** e menor capacidade de generalizar corretamente para o Grupo B.

Em contextos reais, esse tipo de viés pode levar a **conclusões distorcidas e injustas**, reforçando desigualdades sociais e comprometendo a validade dos resultados.  

Exemplo com **banco de dados de aves**:
* A **população real** é composta por **várias espécies de pássaros**, mas as **fotos coletadas são majoritariamente de regiões urbanas**, por questões de acessibilidade, onde há **maior presença de pombos e pardais**

* Nesse cenário, o **modelo treinado** com esses dados **aprenderá a reconhecer muito bem essas espécies urbanas comuns**, mas terá **dificuldade em identificar pássaros raros ou que vivem em áreas rurais**.

* Esse **desbalanceamento não ocorre por causa da população em si**, mas pela forma como os dados foram coletados: certos **grupos ficam sub-representados**, e qualquer análise ou modelo construído sobre essa base produzirá **conclusões distorcidas**, já que não reflete a realidade completa

## 2.2 Viés de Medição

In [ ]:
def demonstrar_vies_medicao():
    # Variáveis proxy introduzem discriminação indireta
    np.random.seed(42)

    dados = {
        'habilidade_real': np.random.normal(0, 1, 1000),
        'grupo': np.random.choice([0, 1], 1000, p=[0.7, 0.3])
    }

    df = pd.DataFrame(dados)
…
    print("Viés de medição - Falsos Negativos por Grupo:\n")
    for grupo in [0, 1]:
        mask = df['grupo'] == grupo
        fn = ((df[mask]['deveria_ser_aprovado'] == 1) &
              (df[mask]['aprovado'] == 0)).mean()
        print(f"Grupo {grupo}: {fn:.1%} de falsos negativos")

    return df

df_medicao = demonstrar_vies_medicao()

Viés de medição - Falsos Negativos por Grupo:

Grupo 0: 5.9% de falsos negativos
Grupo 1: 24.6% de falsos negativos


Vamos analisar o **viés de medição**. Imagine que queremos avaliar a habilidade real das pessoas, mas, em vez disso, utilizamos um teste de pontuação que não trata todos os grupos de forma justa.  

###O que foi feito?

1. **Habilidade real**: Cada indivíduo recebe um valor que representa sua habilidade verdadeira, aquilo que gostaríamos de medir.  
2. **Teste enviesado**: Criamos uma variável chamada `pontuacao_teste`, que deveria refletir a habilidade real. Porém, para os indivíduos do **grupo 1**, subtraímos intencionalmente 0.5 da habilidade antes de adicionar a aleatoriedade. Já o **grupo 0** não sofreu essa penalidade.  
3. **Resultado**: Mesmo quando pessoas dos dois grupos possuem a mesma habilidade real, o teste atribui uma pontuação artificialmente menor ao grupo 1, aumentando suas chances de reprovação.  

**Por que isso é um viés de medição?**  
Esse é um caso de **viés de medição** porque o instrumento de avaliação (o teste) não mede a habilidade real de forma igual para todos. Ele **penaliza sistematicamente o grupo 1**, fazendo com que pareçam menos capazes do que realmente são.

Como consequência, o grupo 1 apresenta uma taxa muito maior de **falsos negativos** (pessoas que deveriam ter sido aprovadas com base em sua habilidade real, mas foram reprovadas pelo teste enviesado). Nos resultados, o **grupo 1** teve **24,6% de falsos negativos**, enquanto o **grupo 0** apresentou apenas **5,9%**.  

**Exemplo em uma competição:**
* Uma competição esportiva em que queremos medir a habilidade real dos participantes em corrida de 100 metros
* No entanto, **o cronômetro usado** para medir o tempo tem um **pequeno atraso quando é acionado em uma pista específica**
* Isso significa que os** corredores que largam nessa pista sempre recebem uma pontuação artificialmente pior**, mesmo que sua velocidade real seja igual à dos demais

> Esse é um bom exemplo, porque **o cronômetro não mede a habilidade de forma justa para todos**. O resultado é que **alguns atletas parecem menos capazes do que realmente são**, não por causa de sua performance, mas por causa da forma enviesada como a medição foi feita

## 2.3 Viés Algorítmico

In [ ]:
def demonstrar_vies_algoritmico():
    from sklearn.linear_model import LogisticRegression

    np.random.seed(42)

    # Dados com padrões diferentes por grupo
    X = np.random.normal(0, 1, (1000, 2))
    grupos = np.random.choice([0, 1], 1000, p=[0.6, 0.4])

    y = np.zeros(1000)

    # Grupo 0: relação linear
    mask_0 = grupos == 0
    y[mask_0] = (X[mask_0, 0] + X[mask_0, 1] > 0).astype(int)

    # Grupo 1: relação quadrática (o modelo linear não consegue capturar)
    mask_1 = grupos == 1
    y[mask_1] = (X[mask_1, 0]**2 + X[mask_1, 1]**2 > 1).astype(int)

    # Modelo linear (incorreto para grupo 1)
    X_com_grupo = np.column_stack([X, grupos])
    modelo = LogisticRegression()
    modelo.fit(X_com_grupo, y)

    predicoes = modelo.predict(X_com_grupo)

    print("Viés algorítmico \n-> Desempenho por grupo:")
    for grupo in [0, 1]:
        mask = grupos == grupo
        acuracia = (predicoes[mask] == y[mask]).mean()
        print(f"Grupo {grupo}: {acuracia:.1%} de acurácia")

    return modelo, X_com_grupo, y, grupos

modelo, X_algo, y_algo, grupos_algo = demonstrar_vies_algoritmico()

Viés algorítmico 
-> Desempenho por grupo:
Grupo 0: 98.3% de acurácia
Grupo 1: 47.3% de acurácia


### O que fizemos?
O desafio era prever um resultado (y) com base em algumas informações (X).

- **Grupo 0**: a relação entre as variáveis é linear (a soma das duas variáveis define o resultado).  
- **Grupo 1**: a relação é quadrática (o resultado depende do quadrado das variáveis), algo que um modelo linear não consegue capturar corretamente.  

Em seguida, treinamos um **modelo de regressão logística**, que é **linear** por natureza.

Esse modelo consegue representar **bem o padrão** do **grupo 0**, mas **não consegue** aprender a **relação mais complexa** do **grupo 1**.

Como consequência, a acurácia do modelo é **alta para o grupo 0 e significativamente menor para o grupo 1.**

### Por que isso é um viés algorítmico?  
Esse é um caso de **viés algorítmico** porque o modelo escolhido (linear) **não é igualmente adequado para todos os grupos**. Ele favorece o grupo cujo padrão se **ajusta melhor à sua estrutura matemática** e prejudica o outro grupo, mesmo que ambos tenham dados suficientes.
> **Na prática, isso significa que o algoritmo não trata todos os grupos de forma justa, produzindo resultados desbalanceados e distorcidos.**

Exemplo em **sistemas de recomendação** (como o da amazon):

Um algoritmo simples baseado em correlação linear funciona bem para usuários que seguem padrões de consumo previsíveis, mas falha para aqueles que têm hábitos mais complexos ou variados.

O resultado é que o sistema **recomenda bons produtos para um grupo de usuários**, mas gera **sugestões irrelevantes para outro**, criando um viés algorítmico.  

##3. Exemplos reais
###3.1 Justiça Criminal - COMPAS

In [ ]:
def exemplo_compas():
    print("Sistema usado para avaliar risco de reincidência criminal")
    print("PROBLEMA: African-Americans (negros) recebiam scores 45% mais altos")
    print("IMPACTO: Pessoas negras tinham ~2x mais falsos positivos")
    print("CAUSA? Dados históricos de policiamento enviesado")

    # Simulação
    np.random.seed(42)
    dados_compas = {
        'raca': np.random.choice(['white', 'black'], 2000, p=[0.5, 0.5]),
        'priors_count': np.random.poisson(2, 2000)
    }

    df_compas = pd.DataFrame(dados_compas)

    # Viés: mesmo número de histórico criminal, score diferente por raça
    df_compas['risk_score'] = (
        df_compas['priors_count'] * 2 +
        (df_compas['raca'] == 'black') * 1.5 +  # Viés sistêmico
        np.random.normal(0, 1, 2000)
    )

    print("\nSimulação COMPAS - Score Médio de risco de reincidência por raça:")
    print(df_compas.groupby('raca').mean())

exemplo_compas()

Sistema usado para avaliar risco de reincidência criminal
PROBLEMA: African-Americans (negros) recebiam scores 45% mais altos
IMPACTO: Pessoas negras tinham ~2x mais falsos positivos
CAUSA? Dados históricos de policiamento enviesado

Simulação COMPAS - Score Médio de risco de reincidência por raça:
       priors_count  risk_score
raca                           
black      2.010859    5.502729
white      1.930091    3.852965


Este exemplo demonstra o caso do **COMPAS**, um sistema real utilizado para avaliar o **risco de reincidência criminal**. O problema identificado foi que pessoas negras recebiam, em média, scores de risco **45% mais altos** do que pessoas brancas, mesmo quando tinham **o mesmo histórico criminal.**

Isso resultava em um **impacto direto:** indivíduos **negros** tinham aproximadamente **2 vezes mais falsos positivos**, ou seja, eram classificados como de **alto risco** sem realmente reincidir e consequentemente **perpetuando ciclos de encarceramento** e dificultando a reintegração social do grupo

**Na simulação**, criamos uma **base de dados fictícia** com duas variáveis principais: a raça da pessoa (`white` ou `black`) e o número de ocorrências anteriores (`priors_count`).

* Em seguida, calculamos um score de risco que **deveria refletir apenas o histórico criminal**.
* No entanto, introduzimos um **viés sistêmico**: para pessoas negras, o score foi aumentado artificialmente em 1.5 pontos. Assim, mesmo com o mesmo número de ocorrências anteriores, **o resultado final penaliza injustamente esse grupo.**  

###Por que isso é um viés?  
Esse é um caso de **viés algorítmico alimentado por dados históricos enviesados**. O problema não está na habilidade do modelo em si, mas nos dados e nas regras que foram utilizadas para calcular o score.

Como os **dados refletem práticas de policiamento historicamente desbalanceadas**, o sistema **reproduz e amplifica essas desigualdades**. Na prática, isso significa que o algoritmo **não mede de forma justa o risco de reincidência**, prejudicando sistematicamente um grupo em relação ao outro.  

###3.2 Sistemas de crédito

In [ ]:
def exemplo_credito():

    print("PROBLEMA: Residentes da região B recebem menos crédito, mesmo tendo os mesmos dados financeiros que região A")
    print("IMPACTO: Limite consideravelmente menor para os residentes de B")
    print("CAUSA? A região do CEP B fica na periferia\n")
    np.random.seed(42)

    # Simulação de dados de clientes
    dados_credito = {
        'renda': np.random.exponential(50000, 1000), # 1500 valores aleatórios com média em torno de 50.000
        'score_credito': np.random.normal(700, 50, 1000), # score médio de 700
        'cep': np.random.choice(['A', 'B'], 1000, p=[0.5, 0.5])  # distribuição igual
    }

    df = pd.DataFrame(dados_credito)
    # clientes do CEP B recebem uma penalidade fixa, mesmo com mesma renda/score
    df['limite_aprovado'] = (
        df['renda'] * 0.4 +
        df['score_credito'] * 8 -
        (df['cep'] == 'B') * 10000   # penalidade injusta para CEP B
    )

    print("Limite médio por CEP:")
    print(df.groupby('cep').mean())

    return df

df_credito = exemplo_credito()

PROBLEMA: Residentes da região B recebem menos crédito, mesmo tendo os mesmos dados financeiros que região A
IMPACTO: Limite consideravelmente menor para os residentes de B
CAUSA? A região do CEP B fica na periferia

Limite médio por CEP:
            renda  score_credito  limite_aprovado
cep                                              
A    50869.240568     703.848216     25978.481954
B    46289.764711     706.086124     14164.594877


Simulamos um viés em **sistemas de crédito** onde, na prática, residentes do CEP B recebem créditos menores, mesmo com dados históricos iguais.

* A simulação cria uma base de dados fictícia com informações de renda, score de crédito e CEP de 1000 clientes.
* O limite aprovado **deveria depender apenas da renda e do score**, mas o cálculo introduz uma regra enviesada: clientes do **CEP B recebem uma penalidade fixa** de 10.000 no limite final, independentemente de suas características financeiras, apenas por ser uma **região periférica**, por exemplo.

> Na prática, isso significa que duas pessoas com a mesma renda e o mesmo score de crédito recebem limites diferentes apenas por causa do CEP. O resultado é que os clientes do CEP B acabam com limites médios significativamente menores, criando uma **desvantagem econômica sistemática.**

### Por que isso é um viés?  
A Partir do momento em que o **critério de decisão não trata todos os grupos de forma justa**. O modelo penaliza sistematicamente um grupo com base em uma variável irrelevante (CEP), **reproduzindo desigualdades que não têm relação direta com a capacidade de pagamento**.

Em contextos reais, esse tipo de regra pode surgir quando algoritmos utilizam proxies inadequados ou dados históricos enviesados, levando a decisões injustas e discriminatórias.  

###3.3 Saúde - Diagnósticos

In [ ]:

def exemplo_saude():

    print("PROBLEMA: espécies diferentes tem reações distintas aos mesmos sintomas")
    print("IMPACTO: Desigualdade em resultados de diagnóstico")
    print("CAUSA: modelo usado no diagnóstico não abraça a complexidade matemática da espécie B\n")
    np.random.seed(42)

    # Simulação em pacientes de diferentes espécies distribuídos igualmente
    grupos = np.random.choice(['espécie A', 'espécie B'], 1000, p=[0.50, 0.50])
    sintoma1 = np.random.normal(0, 1, 1000)
    sintoma2 = np.random.normal(0, 1, 1000)

    df = pd.DataFrame({'grupo': grupos, 'sintoma1': sintoma1, 'sintoma2': sintoma2})

    # Diagnóstico real:
    # Espécie A tem manifestação da doença em padrão linear
    mask_maj = df['grupo'] == 'espécie A'
    df.loc[mask_maj, 'doenca_real'] = ((df.loc[mask_maj, 'sintoma1'] + df.loc[mask_maj, 'sintoma2']) > 0).astype(int)

    # Espécie B tem padrão não-linear, os sintomas se comportam de maneira diferente
    mask_min = df['grupo'] == 'espécie B'
    df.loc[mask_min, 'doenca_real'] = ((df.loc[mask_min, 'sintoma1']**2 + df.loc[mask_min, 'sintoma2']**2) > 1).astype(int)

    # treinamento com todos os dados, mas usando apenas um modelo linear

    modelo = LogisticRegression()
    modelo.fit(df[['sintoma1', 'sintoma2']], df['doenca_real'])

    predicoes = modelo.predict(df[['sintoma1', 'sintoma2']])

    print("Acurácia por grupo:")
    for g in ['espécie A', 'espécie B']:
        mask = df['grupo'] == g
        acc = (predicoes[mask] == df.loc[mask, 'doenca_real']).mean()
        print(f"{g}: {acc:.1%}")

    return df, modelo

df_saude, modelo_saude = exemplo_saude()

PROBLEMA: espécies diferentes tem reações distintas aos mesmos sintomas
IMPACTO: Desigualdade em resultados de diagnóstico
CAUSA: modelo usado no diagnóstico não abraça a complexidade matemática da espécie B

Acurácia por grupo:
espécie A: 94.8%
espécie B: 50.7%


Demonstramos um caso de **viés em sistemas de saúde e diagnóstico**.

A simulação cria **pacientes de duas espécies diferentes**, cada uma apresentando **padrões distintos de manifestação da doença**.  

- **Espécie A**: a relação entre os sintomas e a doença segue um padrão **linear**, que pode ser facilmente capturado por modelos simples.  
- **Espécie B**: a relação é **não-linear**, ou seja, os sintomas se combinam de forma mais complexa, exigindo modelos mais sofisticados para serem corretamente interpretados.  

O modelo escolhido para o diagnóstico foi uma **regressão logística linear**, treinada com dados de ambas as espécies.

> Como consequência, ele consegue **representar bem os padrões da espécie A**, mas **falha em capturar a complexidade da espécie B**.  

### Por que isso é um viés?  
Esse é um caso de **viés algorítmico** porque o modelo não trata todos os grupos de forma justa. Ele favorece a espécie cujo padrão se ajusta melhor à sua estrutura matemática e prejudica a outra, mesmo que ambas tenham dados suficientes.

Na prática, isso significa que **pacientes da espécie B recebem diagnósticos menos precisos**, criando uma desigualdade nos resultados de saúde e prejudicando tratamentos adequados